## Session 4 — High Level Flow

### Step 1 — Setup
- Install libraries: unsloth, huggingface_hub
- Login to HuggingFace using API token
- Load API keys from Colab Secrets

### Step 2 — Recreate fine-tuned model
- Load base model (Llama 3.2 1B)
- Attach LoRA adapters
- Fine-tune on swiggy_train.jsonl
- (same as Session 2 — needed because Colab resets)

### Step 3 — Save LoRA adapters locally
- Save ONLY the adapter weights to disk
- NOT the full 1B model — just the small adapter file
- Folder: swiggy-support-adapter/

### Step 4 — Push to HuggingFace Hub
- Create a public repo on HuggingFace
- Upload adapter weights to the repo
- Get a public URL: huggingface.co/your-username/swiggy-support-adapter

### Step 5 — Load adapter back from Hub
- Load base model fresh
- Pull adapter weights from HuggingFace URL
- Merge adapter into model for inference

### Step 6 — Test inference
- Give the loaded model a new Swiggy complaint
- Confirm it still responds like a trained support agent
- This proves the save → push → load cycle works end to end

# Module 4 — Session 4: Save Adapters & Push to HuggingFace Hub
## Goal: Save LoRA adapter weights, push to HuggingFace Hub, and load back for inference

In [ ]:
# Install required libraries
# huggingface_hub — lets us push and pull models from HuggingFace
!pip install unsloth huggingface_hub -q

In [ ]:
# Verify both libraries installed correctly
import unsloth
import huggingface_hub

print(f"Unsloth version: {unsloth.__version__}")
print(f"HuggingFace Hub version: {huggingface_hub.__version__}")
print("All libraries ready!")

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login

# Load HuggingFace token from Colab Secrets
hf_token = userdata.get("HF_TOKEN")

# Login to HuggingFace Hub
login(token=hf_token)

print("HuggingFace login successful!")

In [ ]:
!pip install groq -q

In [ ]:
import os
import json
import random
from groq import Groq
from google.colab import userdata

# Load Groq API key
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
client = Groq()

print("Groq client ready!")

In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# Step 1 — Load base model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = 512,
    load_in_4bit = True
)

# Step 2 — Attach LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
)

print("Base model + LoRA adapters ready!")

In [ ]:
# Recreate seed examples
seed_examples = [
    {
        "instruction": "You are a Swiggy customer support agent. Be empathetic, helpful, and always offer a solution.",
        "input": "It's been over an hour and my order hasn't arrived. I'm very hungry!",
        "output": "I sincerely apologize for this delay! I've tracked your order and see the delivery partner is 8 minutes away due to heavy rain. I've added ₹75 to your Swiggy wallet for this inconvenience. We value your patience!"
    },
    {
        "instruction": "You are a Swiggy customer support agent. Be empathetic, helpful, and always offer a solution.",
        "input": "I ordered a Veg Burger but received a Chicken Burger. I am vegetarian!",
        "output": "I'm truly sorry for this serious mistake! I've raised a complaint with the restaurant and initiated a full refund of ₹249 within 5-7 business days. We take dietary requirements very seriously."
    },
    {
        "instruction": "You are a Swiggy customer support agent. Be empathetic, helpful, and always offer a solution.",
        "input": "My order was cancelled automatically but money was deducted from my account.",
        "output": "I apologize for this inconvenience! Your payment of ₹380 will be refunded within 5-7 business days. I've added ₹50 to your Swiggy wallet to compensate. Please feel free to reorder!"
    },
    {
        "instruction": "You are a Swiggy customer support agent. Be empathetic, helpful, and always offer a solution.",
        "input": "My order arrived but the french fries were missing. I paid ₹80 extra for them!",
        "output": "I'm sorry about the missing fries! I've verified your order and refunded ₹80 to your Swiggy wallet immediately. Thank you for flagging this!"
    },
    {
        "instruction": "You are a Swiggy customer support agent. Be empathetic, helpful, and always offer a solution.",
        "input": "The delivery partner was very rude to me and threw the food bag at my door.",
        "output": "I sincerely apologize for this unacceptable behavior! I've flagged this for immediate review. I've added ₹150 to your Swiggy wallet as compensation. You deserve respectful service always!"
    }
]

# Generate synthetic examples
categories = ["order_delay", "wrong_item", "refund_request",
              "missing_item", "app_issue", "delivery_behavior"]

def generate_examples_for_category(category, n=4):
    prompt = f"""You are creating training data for a Swiggy customer support AI.

Generate exactly {n} different customer support examples for the category: {category}

Rules:
- instruction must always be: "You are a Swiggy customer support agent. Be empathetic, helpful, and always offer a solution."
- input must be a realistic Indian customer complaint (1-2 sentences)
- output must be an empathetic agent response (3-4 sentences, always offer a solution)
- Use Indian context: mention ₹ amounts, Indian cities like Mumbai/Delhi/Bengaluru/Chennai

Return ONLY a valid JSON array, no explanation, no markdown, no extra text.
Format:
[
  {{"instruction": "...", "input": "...", "output": "..."}},
  {{"instruction": "...", "input": "...", "output": "..."}}
]"""
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.8
    )
    return json.loads(response.choices[0].message.content)

# Generate and save
all_examples = seed_examples.copy()
for category in categories:
    print(f"Generating: {category}...")
    try:
        new_examples = generate_examples_for_category(category, n=4)
        all_examples.extend(new_examples)
    except Exception as e:
        print(f"  Error: {e}")

# Shuffle and save
random.seed(42)
random.shuffle(all_examples)
train_data = all_examples[:int(len(all_examples) * 0.8)]

with open("swiggy_train.jsonl", "w") as f:
    for item in train_data:
        f.write(json.dumps(item) + "\n")

print(f"\nTotal examples: {len(all_examples)}")
print(f"Train examples saved: {len(train_data)}")

In [ ]:
# Format dataset and fine-tune
alpaca_prompt = """Below is an instruction with an input. Write a response.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

# Load and format training data
train_data_loaded = []
with open("swiggy_train.jsonl", "r") as f:
    for line in f:
        train_data_loaded.append(json.loads(line.strip()))

formatted_texts = []
for ex in train_data_loaded:
    text = alpaca_prompt.format(
        ex["instruction"], ex["input"], ex["output"]
    ) + tokenizer.eos_token
    formatted_texts.append({"text": text})

train_dataset = Dataset.from_list(formatted_texts)

# Fine-tune
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = 512,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        output_dir = "outputs",
        logging_steps = 1,
        save_strategy = "no",
    ),
)

trainer.train()
print("Fine-tuning complete!")

In [ ]:
# Save ONLY the LoRA adapter weights to a local folder
# NOT the full model — just the small adapter file
# This is the key advantage of LoRA — adapter is only a few MB vs full model which is GBs

adapter_path = "swiggy-support-adapter"

model.save_pretrained(adapter_path)   # saves LoRA adapter weights
tokenizer.save_pretrained(adapter_path)  # saves tokenizer too

print(f"Adapter saved to: {adapter_path}")

# Check what files were saved
import os
files = os.listdir(adapter_path)
print(f"\nFiles saved:")
for f in files:
    size = os.path.getsize(f"{adapter_path}/{f}")
    print(f"  {f} — {size/1024:.1f} KB")

In [ ]:
# Push adapter weights to HuggingFace Hub
# This creates a public repo and uploads all adapter files

repo_name = "zaf786/swiggy-support-adapter"

model.push_to_hub(repo_name, token=hf_token)      # push adapter weights
tokenizer.push_to_hub(repo_name, token=hf_token)  # push tokenizer

print(f"\nAdapter pushed successfully!")
print(f"Public URL: https://huggingface.co/{repo_name}")

## Step 5 — Load Adapter Back from HuggingFace Hub
Loading the adapter from HuggingFace Hub proves the full cycle works:
save locally → push to Hub → load back from Hub → run inference.
This is how any developer in the world can use your fine-tuned model.

In [ ]:
# Load base model fresh
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = 512,
    load_in_4bit = True
)

# Pull LoRA adapter from HuggingFace Hub and merge into base model
# PeftModel is the HuggingFace library for loading LoRA adapters
from peft import PeftModel

loaded_model = PeftModel.from_pretrained(
    base_model,
    "zaf786/swiggy-support-adapter",  # your public HuggingFace repo
    token = hf_token
)

print("Adapter loaded from HuggingFace Hub successfully!")

## Step 6 — Test Inference
Sending a brand new Swiggy complaint to the model loaded from HuggingFace Hub.
This confirms the full pipeline works end to end — the model still behaves
like a trained Swiggy support agent even after save → push → load.

In [ ]:
# Switch to inference mode
FastLanguageModel.for_inference(loaded_model)

# Brand new complaint — never seen during training
test_complaint = "I ordered paneer butter masala from a Delhi restaurant but it arrived completely spilled. The packaging was broken!"

# Format using Alpaca template
prompt = alpaca_prompt.format(
    "You are a Swiggy customer support agent. Be empathetic, helpful, and always offer a solution.",
    test_complaint,
    ""  # empty — model fills this in
)

# Tokenize and move to GPU
inputs = base_tokenizer(prompt, return_tensors="pt").to("cuda")

# Generate response
outputs = loaded_model.generate(**inputs, max_new_tokens=150)

# Decode and extract only the response part
full_response = base_tokenizer.decode(outputs[0], skip_special_tokens=True)
response = full_response.split("### Response:")[-1].strip()

print("COMPLAINT:")
print(test_complaint)
print("\nFINE-TUNED MODEL RESPONSE:")
print(response)

## What We Learned Today

### New Python Concepts
- `model.save_pretrained()` — saves LoRA adapter weights to local folder
- `tokenizer.save_pretrained()` — saves tokenizer files to local folder
- `model.push_to_hub()` — uploads adapter to HuggingFace Hub (public URL)
- `PeftModel.from_pretrained()` — loads LoRA adapter from HuggingFace Hub
- `os.listdir()` — lists all files in a folder
- `os.path.getsize()` — gets file size in bytes

### Key Insight
LoRA adapter is only 43 MB vs full model of 2 GB — 50x smaller!
This is why LoRA is the industry standard for fine-tuning.

### AWS Equivalent
- Saving adapter locally → **S3** (store adapter weights)
- Pushing to HuggingFace Hub → **SageMaker Model Registry** (version and store models)
- Loading adapter from Hub → **SageMaker Endpoints** (deploy and serve model)
- Full pipeline → **SageMaker Pipelines** (automate train → save → deploy)

### Public Model URL
https://huggingface.co/zaf786/swiggy-support-adapter